In [ ]:
from google.colab import files
import zipfile
import os

# Upload the zip file
uploaded = files.upload()

# Extract the zip file
for filename in uploaded.keys():
    if filename.endswith('.zip'):  # Ensure it's a zip file
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall("data")  # Extracts all contents into "data" folder
        print(f"Extracted: {filename}")
    else:
        print(f"Uploaded file is not a zip file: {filename}")

# Check the contents of the extracted folder
print("Contents of the 'data' folder:")
print(os.listdir("data"))


Saving isolated_words_per_user.zip to isolated_words_per_user.zip
Extracted: isolated_words_per_user.zip
Contents of the 'data' folder:
['isolated_words_per_user']


In [ ]:
import cv2
import glob
import os
import matplotlib.pyplot as plt

# Define the path to the extracted folder
base_path = "data/isolated_words_per_user"

# Initialize lists to store all images and labels
all_images = []
all_labels = []

# Sort folders for consistent ordering
sorted_folders = sorted(os.listdir(base_path))

# Loop through all sorted folders and store images
for folder in sorted_folders:
    folder_path = os.path.join(base_path, folder)  # Path to each user's folder

    if os.path.isdir(folder_path):  # Ensure it's a directory
        # Load all image paths in the folder and sort them
        images_in_folder = sorted(glob.glob(folder_path + "/*.png"))

        for file in images_in_folder:
            img = cv2.imread(file, cv2.IMREAD_GRAYSCALE)  # Read the image in grayscale
            if img is not None:  # Ensure the image is valid
                all_images.append(img)
                all_labels.append(folder)  # Store the folder name as the label

# Print summary
print(f"Loaded {len(all_images)} images from {len(set(all_labels))} folders.")

# Display images from the first 3 sorted folders
for folder in sorted_folders[:3]:  # First 3 folders
    folder_path = os.path.join(base_path, folder)
    print(f"\nDisplaying all images from folder: {folder}")

    # Load and sort images within the folder
    images_in_folder = sorted(glob.glob(folder_path + "/*.png"))

    for i, file in enumerate(images_in_folder):
        img = cv2.imread(file, cv2.IMREAD_GRAYSCALE)  # Read the image in grayscale
        if img is not None:
            plt.figure(figsize=(6, 6))
            plt.imshow(img, cmap='gray')
            plt.title(f"Folder: {folder} | Image {i+1}")
            plt.axis('off')
            plt.show()
        else:
            print(f"Failed to load image: {file}")


In [ ]:
import cv2
import os
import glob
import matplotlib.pyplot as plt

# Define the path to the dataset
base_path = "data/isolated_words_per_user"

# Initialize lists to store all keypoints and descriptors
all_keypoints = []  # List to store keypoints for all images
all_descriptors = []  # List to store descriptors for all images

# Create a SIFT object
sift = cv2.SIFT_create()

# Apply SIFT to all images in the dataset with progress updates
print("\nApplying SIFT to the dataset with progress updates...")

# Process each folder in the dataset
total_images_processed = 0  # Counter for processed images
for folder_idx, folder in enumerate(sorted(os.listdir(base_path))):  # Use sorted() for consistent order
    folder_path = os.path.join(base_path, folder)  # Full path to the folder

    if os.path.isdir(folder_path):  # Ensure it is a folder
        # Process all images inside the folder
        for file_idx, file in enumerate(sorted(glob.glob(folder_path + "/*.png"))):  # Sort images for consistency
            img = cv2.imread(file, cv2.IMREAD_GRAYSCALE)  # Read the image in grayscale
            if img is not None:  # Ensure the image is valid
                # Extract keypoints and descriptors
                keypoints, descriptors = sift.detectAndCompute(img, None)
                all_keypoints.append(keypoints)  # Store keypoints
                all_descriptors.append(descriptors)  # Store descriptors

                total_images_processed += 1

                # Print progress every 50 images
                if total_images_processed % 50 == 0:
                    print(f"Processed {total_images_processed} images...")

print(f"\nFinished processing all images. Total images processed: {total_images_processed}")

# Display results for the first 3 folders and all images inside them
print("\nDisplaying SIFT features for the first 3 folders:")

folders = sorted(os.listdir(base_path))[:3]  # Get the first 3 folders in sorted order
for folder in folders:
    folder_path = os.path.join(base_path, folder)  # Path to the folder
    print(f"\nFolder: {folder}")

    # Process all images inside the folder
    images_in_folder = sorted(glob.glob(folder_path + "/*.png"))  # Get all images in the folder in sorted order
    for i, file in enumerate(images_in_folder):
        img = cv2.imread(file, cv2.IMREAD_GRAYSCALE)  # Read the image in grayscale
        if img is not None:
            keypoints, descriptors = sift.detectAndCompute(img, None)  # Extract keypoints and descriptors

            # Draw keypoints on the image
            img_with_keypoints = cv2.drawKeypoints(
                img, keypoints, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
            )
            plt.figure(figsize=(6, 6))
            plt.imshow(img_with_keypoints, cmap='gray')
            plt.title(f"Folder: {folder} | Image {i+1} | Keypoints: {len(keypoints)}")
            plt.axis('off')
            plt.show()

            # Print details for the image
            print(f"Image {i+1}:")
            print(f"- Number of keypoints: {len(keypoints)}")
            print(f"- Descriptors shape: {descriptors.shape if descriptors is not None else 'None'}")
            print("-" * 40)

print("\nSIFT features extracted and visualized for the first 3 folders.")


In [ ]:
import numpy as np
from sklearn.cluster import MiniBatchKMeans
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import cv2
import time
import glob
import os

# Step 1: Combine Descriptors for Clustering
print("\nCombining SIFT descriptors for clustering...")
all_descriptors_combined = [desc for desc in all_descriptors if desc is not None]  # Exclude None descriptors

if len(all_descriptors_combined) == 0:
    raise ValueError("No valid descriptors found from SIFT. Ensure the SIFT extraction was successful.")

# Stack all descriptors into one array for clustering
all_descriptors_combined = np.vstack(all_descriptors_combined)

# Step 2: Perform MiniBatch K-Means Clustering
n_clusters = 100  # Number of visual words (clusters)
print(f"\nPerforming MiniBatch K-Means clustering with {n_clusters} clusters...")
start_time_kmeans = time.time()
kmeans = MiniBatchKMeans(n_clusters=n_clusters, batch_size=1000, random_state=42)
kmeans.fit(all_descriptors_combined)
end_time_kmeans = time.time()
print(f"K-Means clustering completed in {end_time_kmeans - start_time_kmeans:.2f} seconds.")

# Step 3: Create Feature Vectors (Histograms) for Each Image
print("\nCreating feature vectors (histograms) for each image...")
feature_vectors = []
for descriptors in all_descriptors:
    if descriptors is not None:
        histogram = np.zeros(n_clusters)
        cluster_indices = kmeans.predict(descriptors)
        for idx in cluster_indices:
            histogram[idx] += 1
        feature_vectors.append(histogram)
    else:
        feature_vectors.append(np.zeros(n_clusters))  # Empty histogram for images without descriptors

# Normalize feature vectors
X = np.array(feature_vectors)
y = np.array(labels)  # Labels from the SIFT extraction step
scaler = StandardScaler()
start_time_scaling = time.time()
X = scaler.fit_transform(X)
end_time_scaling = time.time()

print(f"Feature vectors created with shape: {X.shape}")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"Training set size: {len(X_train)}, Testing set size: {len(X_test)}")

# Step 4: Train and Evaluate SVM Classifier
print("\nTraining SVM Classifier...")
start_time_svm = time.time()
svm = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
svm.fit(X_train, y_train)
end_time_svm = time.time()
print("SVM model trained successfully.")

# Evaluate the SVM model
start_time_prediction = time.time()
y_pred = svm.predict(X_test)
end_time_prediction = time.time()
accuracy = accuracy_score(y_test, y_pred)
classification = classification_report(y_test, y_pred, zero_division=0)

print(f"\nSVM Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification)

# Step 5: Calculate Average Key Points
average_keypoints = np.mean([len(kp) for kp in all_keypoints])
print(f"Average Keypoints Detected: {average_keypoints:.2f}")

# Step 6: Robustness Testing
print("\n--- Robustness Results ---")
transformations = {
    "Scale 0.5": lambda img: cv2.resize(img, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_LINEAR),
    "Rotate 45°": lambda img: cv2.warpAffine(
        img, cv2.getRotationMatrix2D((img.shape[1] // 2, img.shape[0] // 2), 45, 1), (img.shape[1], img.shape[0])
    ),
    "Gaussian Noise": lambda img: np.clip(img + np.random.normal(0, 25, img.shape).astype(np.uint8), 0, 255),
}

robustness_results = {transform: [] for transform in transformations.keys()}

# Iterate over all images in the dataset
all_image_paths = []
for folder in os.listdir("data/isolated_words_per_user"):
    folder_path = os.path.join("data/isolated_words_per_user", folder)
    if os.path.isdir(folder_path):
        for file in glob.glob(os.path.join(folder_path, "*.png")):
            img = cv2.imread(file, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            all_image_paths.append(file)  # Collect valid paths
            for transform_name, transform_fn in transformations.items():
                transformed_img = transform_fn(img)
                sift = cv2.SIFT_create()
                kp, _ = sift.detectAndCompute(transformed_img, None)
                robustness_results[transform_name].append(len(kp))

# Compute average keypoints for each transformation
for transform_name, keypoints_list in robustness_results.items():
    robustness_results[transform_name] = np.mean(keypoints_list)
    print(f"{transform_name}: {robustness_results[transform_name]:.2f} keypoints detected on average.")

# Step 7: Visualize All Images from Folder user002 with Transformations
print("\n--- Visualizing All Images from Folder: user002 ---")

user002_folder = os.path.join("data/isolated_words_per_user", "user002")
if not os.path.isdir(user002_folder):
    raise ValueError("Folder user002 not found. Ensure the dataset contains this folder.")

# Get all image paths from user002 folder
user002_image_paths = sorted(glob.glob(os.path.join(user002_folder, "*.png")))

if not user002_image_paths:
    raise ValueError("No images found in folder user002.")

for sample_image_path in user002_image_paths:
    img = cv2.imread(sample_image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"Could not load {sample_image_path}")
        continue

    print(f"\nVisualizing transformations for: {sample_image_path}")
    fig, axes = plt.subplots(1, len(transformations) + 1, figsize=(20, 6))

    # Display the original image with keypoints
    sift = cv2.SIFT_create()
    kp, _ = sift.detectAndCompute(img, None)
    img_with_keypoints = cv2.drawKeypoints(
        img, kp, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
    )
    axes[0].imshow(img_with_keypoints, cmap='gray')
    axes[0].set_title(f"Original\nKeypoints: {len(kp)}")
    axes[0].axis('off')

    # Display the transformed images with keypoints
    for ax, (transform_name, transform_fn) in zip(axes[1:], transformations.items()):
        transformed_img = transform_fn(img)
        kp, _ = sift.detectAndCompute(transformed_img, None)
        img_with_keypoints = cv2.drawKeypoints(
            transformed_img, kp, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
        )
        ax.imshow(img_with_keypoints, cmap='gray')
        ax.set_title(f"{transform_name}\nKeypoints: {len(kp)}")
        ax.axis('off')

    plt.suptitle(f"Visualizations for {sample_image_path}", fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import os
import glob
import matplotlib.pyplot as plt

# Define the path to the dataset
base_path = "data/isolated_words_per_user"

# Initialize lists to store all keypoints and descriptors
all_orb_keypoints = []  # List to store keypoints for all images
all_orb_descriptors = []  # List to store descriptors for all images

# Create an ORB object
orb = cv2.ORB_create()

# Apply ORB to all images in the dataset with progress updates
print("\nApplying ORB to the dataset with progress updates...")

# Process each folder in the dataset
total_images_processed = 0  # Counter for processed images
for folder_idx, folder in enumerate(sorted(os.listdir(base_path))):  # Use sorted() for consistent order
    folder_path = os.path.join(base_path, folder)  # Full path to the folder

    if os.path.isdir(folder_path):  # Ensure it is a folder
        # Process all images inside the folder
        for file_idx, file in enumerate(sorted(glob.glob(folder_path + "/*.png"))):  # Sort images for consistency
            img = cv2.imread(file, cv2.IMREAD_GRAYSCALE)  # Read the image in grayscale
            if img is not None:  # Ensure the image is valid
                # Extract keypoints and descriptors using ORB
                keypoints, descriptors = orb.detectAndCompute(img, None)
                all_orb_keypoints.append(keypoints)  # Store keypoints
                all_orb_descriptors.append(descriptors)  # Store descriptors

                total_images_processed += 1

                # Print progress every 50 images
                if total_images_processed % 50 == 0:
                    print(f"Processed {total_images_processed} images...")

print(f"\nFinished processing all images with ORB. Total images processed: {total_images_processed}")

# Display results for the first 3 folders and all images inside them
print("\nDisplaying ORB features for the first 3 folders:")

folders = sorted(os.listdir(base_path))[:3]  # Get the first 3 folders in sorted order
for folder in folders:
    folder_path = os.path.join(base_path, folder)  # Path to the folder
    print(f"\nFolder: {folder}")

    # Process all images inside the folder
    images_in_folder = sorted(glob.glob(folder_path + "/*.png"))  # Get all images in the folder in sorted order
    for i, file in enumerate(images_in_folder):
        img = cv2.imread(file, cv2.IMREAD_GRAYSCALE)  # Read the image in grayscale
        if img is not None:
            keypoints, descriptors = orb.detectAndCompute(img, None)  # Extract keypoints and descriptors

            # Draw keypoints on the image
            img_with_keypoints = cv2.drawKeypoints(
                img, keypoints, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
            )
            plt.figure(figsize=(6, 6))
            plt.imshow(img_with_keypoints, cmap='gray')
            plt.title(f"Folder: {folder} | Image {i+1} | Keypoints: {len(keypoints)}")
            plt.axis('off')
            plt.show()

            # Print details for the image
            print(f"Image {i+1}:")
            print(f"- Number of keypoints: {len(keypoints)}")
            print(f"- Descriptors shape: {descriptors.shape if descriptors is not None else 'None'}")
            print("-" * 40)

print("\nORB features extracted and visualized for the first 3 folders.")


In [ ]:
import numpy as np
import time
from sklearn.cluster import MiniBatchKMeans
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import cv2
import os
import glob
import matplotlib.pyplot as plt

# Combine ORB descriptors
print("\nCombining ORB descriptors for clustering...")

# Stack all valid ORB descriptors
start_time_extraction = time.time()
all_descriptors_combined = [desc for desc in all_orb_descriptors if desc is not None]
if not all_descriptors_combined:
    raise ValueError("No valid descriptors found from ORB extraction.")

all_descriptors_combined = np.vstack(all_descriptors_combined)

# Perform MiniBatch K-Means Clustering
n_clusters = 100
print(f"\nPerforming MiniBatch K-Means clustering with {n_clusters} clusters...")
start_time_kmeans = time.time()
kmeans = MiniBatchKMeans(n_clusters=n_clusters, batch_size=1000, random_state=42)
kmeans.fit(all_descriptors_combined)
end_time_kmeans = time.time()
print(f"K-Means clustering completed in {end_time_kmeans - start_time_kmeans:.2f} seconds.")

# Create Feature Vectors (Histograms) for Each Image
print("\nCreating feature vectors (histograms) for each image...")
feature_vectors = []
for descriptors in all_orb_descriptors:
    if descriptors is not None:
        histogram = np.zeros(n_clusters)
        cluster_indices = kmeans.predict(descriptors)
        for idx in cluster_indices:
            histogram[idx] += 1
        feature_vectors.append(histogram)
    else:
        feature_vectors.append(np.zeros(n_clusters))  # Empty histogram for images without descriptors

X = np.array(feature_vectors)
y = np.array(labels)  # Labels for the images

# Normalize Feature Vectors
scaler = StandardScaler()
X = scaler.fit_transform(X)
end_time_extraction = time.time()

print(f"Feature vectors created with shape: {X.shape}")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
print(f"Training set size: {len(X_train)}, Testing set size: {len(X_test)}")

# Train SVM Classifier
print("\nTraining SVM Classifier...")
start_time_training = time.time()
svm = SVC(kernel='rbf', C=10, gamma='scale', random_state=42)
svm.fit(X_train, y_train)
end_time_training = time.time()
print("SVM model trained successfully.")

# Evaluate SVM Classifier
start_time_prediction = time.time()
y_pred = svm.predict(X_test)
end_time_prediction = time.time()
accuracy = accuracy_score(y_test, y_pred)
classification = classification_report(y_test, y_pred, zero_division=0)

print(f"\nSVM Accuracy: {accuracy * 100:.2f}%")
print("\nClassification Report:")
print(classification)

# Calculate Average Key Points
average_keypoints = np.mean([len(kp) for kp in all_orb_keypoints if kp is not None])
print(f"Average Keypoints Detected: {average_keypoints:.2f}")

# Robustness Testing
print("\n--- Robustness Results ---")
transformations = {
    "Scale 0.5": lambda img: cv2.resize(img, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_LINEAR),
    "Rotate 45°": lambda img: cv2.warpAffine(
        img, cv2.getRotationMatrix2D((img.shape[1] // 2, img.shape[0] // 2), 45, 1), (img.shape[1], img.shape[0])
    ),
    "Gaussian Noise": lambda img: np.clip(img + np.random.normal(0, 25, img.shape).astype(np.uint8), 0, 255),
}

robustness_results = {transform: [] for transform in transformations.keys()}

# Iterate over all images in the dataset
all_image_paths = []
for folder in sorted(os.listdir("data/isolated_words_per_user")):
    folder_path = os.path.join("data/isolated_words_per_user", folder)
    if os.path.isdir(folder_path):
        for file in sorted(glob.glob(os.path.join(folder_path, "*.png"))):
            img = cv2.imread(file, cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            all_image_paths.append(file)  # Collect valid paths
            for transform_name, transform_fn in transformations.items():
                transformed_img = transform_fn(img)
                orb = cv2.ORB_create()
                kp, _ = orb.detectAndCompute(transformed_img, None)
                robustness_results[transform_name].append(len(kp))

# Compute Average Key Points for Each Transformation
for transform_name, keypoints_list in robustness_results.items():
    robustness_results[transform_name] = np.mean(keypoints_list)
    print(f"{transform_name}: {robustness_results[transform_name]:.2f} keypoints detected on average.")

# Visualize All Images in user002 Folder with Transformations
print("\n--- Visualizing Transformations for user002 Images ---")
user002_path = os.path.join("data/isolated_words_per_user", "user002")
user002_images = sorted(glob.glob(os.path.join(user002_path, "*.png")))

for img_path in user002_images:
    img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"Could not load {img_path}")
        continue

    print(f"\nVisualizing transformations for: {img_path}")
    fig, axes = plt.subplots(1, len(transformations), figsize=(18, 6))
    for ax, (transform_name, transform_fn) in zip(axes, transformations.items()):
        transformed_img = transform_fn(img)
        orb = cv2.ORB_create()
        kp, _ = orb.detectAndCompute(transformed_img, None)
        img_with_keypoints = cv2.drawKeypoints(
            transformed_img, kp, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS
        )
        ax.imshow(img_with_keypoints, cmap='gray')
        ax.set_title(f"{transform_name}\nKeypoints: {len(kp)}")
        ax.axis('off')

    plt.suptitle(f"Visualizations for {img_path}", fontsize=16)
    plt.tight_layout()
    plt.show()

# Final Metrics Output
print("\n--- Metrics for Comparison ---")
print(f"Time for Feature Extraction: {end_time_extraction - start_time_extraction:.2f} seconds")
print(f"Time for K-Means Clustering: {end_time_kmeans - start_time_kmeans:.2f} seconds")
print(f"Time for Training SVM: {end_time_training - start_time_training:.2f} seconds")
print(f"Time for Prediction: {end_time_prediction - start_time_prediction:.2f} seconds")
print(f"Average Number of Key Points Detected per Image: {average_keypoints:.2f}")
